In [45]:
from collections import Counter

MAX_BYTE_NUM = 256
ENCODE_TYPE = "utf-8"
text = """low low low low low lower lower widest widest widest newest newest newest newest newest newest"""
special_encoded_tokens = ["|endoftext|".encode(ENCODE_TYPE)]

In [87]:
def default_vocab(special_encoded_tokens: list[bytes]):
    vocab = {}
    for index  in range(MAX_BYTE_NUM):
        vocab[index] = chr(index)
    
    for i, special_encoded_token in enumerate(special_encoded_tokens):
        vocab[MAX_BYTE_NUM + i] = special_encoded_token
    
    return vocab

def pre_tokenization(text):
    words = text.split(" ")
    words_counter = Counter(words)
    return dict(word_counts)

def pair_count(cand_list):
    status = Counter()
    for cand_key, cand_value in cand_list.items():
        for char1, char2 in zip(cand_key[:-1], cand_key[1:]):
            chars = (char1, char2)
            status[chars] += cand_value
    
    # 频数排序，相同频数字典序排序，倒序
    status = sorted(
        status.items(),
        key=lambda kv: (kv[1], kv[0]),
        reverse=True
    )

    return dict(status)

def merge(pair_counted, cand_list):
    max_cnt_pair = next(iter(pair_counted)) # 需要合并的pair
    new_cand_list = {}
    for cand_key, cand_value in cand_list.items():
        # max_cnt_pair是否在cand_key中邻近存在
        is_in = any(cand_key[i:i+len(max_cnt_pair)] == max_cnt_pair for i in range(len(cand_key) - len(max_cnt_pair) + 1))
        if is_in:
            new_key = []

            # 合并存在的max_cnt_pair
            i = 0
            while i < len(cand_key):
                if cand_key[i:i+len(max_cnt_pair)]==max_cnt_pair:
                    new_key.append("".join(max_cnt_pair))
                    i += len(max_cnt_pair)
                else:
                    new_key.append(cand_key[i])
                    i += 1
            new_key = tuple(new_key)
            new_cand_list.update({new_key:cand_value})
        else:
            new_cand_list.update({cand_key:cand_value})

    return new_cand_list, max_cnt_pair


In [94]:
vocab_size = MAX_BYTE_NUM + 1 + 6

vocab = default_vocab(special_encoded_tokens)
pre_tokens = pre_tokenization(text)
merges = {tuple(list(k)): v for k, v in pre_tokens.items()}
new_tokens = []

for _ in range(vocab_size - MAX_BYTE_NUM - 1):
    pair_cnt = pair_count(merges)
    # print(pair_cnt)
    merges, new_token = merge(pair_cnt, merges)
    new_tokens.append(new_token)
    print(merges)

print(new_tokens)
new_tokens = ["".join(new_token) for new_token in new_tokens]
for token in new_tokens:
    vocab_size = len(vocab)
    vocab[vocab_size] = token.encode(ENCODE_TYPE)
print(vocab)


{('l', 'o', 'w'): 5, ('l', 'o', 'w', 'e', 'r'): 2, ('w', 'i', 'd', 'e', 'st'): 3, ('n', 'e', 'w', 'e', 'st'): 6}
{('l', 'o', 'w'): 5, ('l', 'o', 'w', 'e', 'r'): 2, ('w', 'i', 'd', 'est'): 3, ('n', 'e', 'w', 'est'): 6}
{('l', 'ow'): 5, ('l', 'ow', 'e', 'r'): 2, ('w', 'i', 'd', 'est'): 3, ('n', 'e', 'w', 'est'): 6}
{('low',): 5, ('low', 'e', 'r'): 2, ('w', 'i', 'd', 'est'): 3, ('n', 'e', 'w', 'est'): 6}
{('low',): 5, ('low', 'e', 'r'): 2, ('w', 'i', 'd', 'est'): 3, ('n', 'e', 'west'): 6}
{('low',): 5, ('low', 'e', 'r'): 2, ('w', 'i', 'd', 'est'): 3, ('ne', 'west'): 6}
[('s', 't'), ('e', 'st'), ('o', 'w'), ('l', 'ow'), ('w', 'est'), ('n', 'e')]
{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8: '\x08', 9: '\t', 10: '\n', 11: '\x0b', 12: '\x0c', 13: '\r', 14: '\x0e', 15: '\x0f', 16: '\x10', 17: '\x11', 18: '\x12', 19: '\x13', 20: '\x14', 21: '\x15', 22: '\x16', 23: '\x17', 24: '\x18', 25: '\x19', 26: '\x1a', 27: '\x1b', 28: '\x1c', 29: '\x1d', 30: '